In [ ]:
import os
import pandas as pd
import numpy as np
from metamoth import parse_metadata
import matplotlib.pyplot as plt

import json
from tqdm import tqdm

## Input/Output Files & Parameters

In [ ]:
# ================== INPUT/OUTPUT FILES ===================

# Folder containing all raw audio splits
SPLITS_FOLDER = r''

# Path to splits breakdown
ALL_SPLITS = r''

# Path to raven selection, used as ground truth label (.txt file)
RAVEN_SEL_PATH = r''

# Path where file info from all manually annotated files is written
MANUAL_FILE_LIST = r''

# ====================== PARAMETERS =======================

SAMPLE_RATE = 48000

# If a peak is below this distance (in s) from a file boundary, it will not
# be included in peak duration statistics
BOUNDARY_BUFFER = 0.5

## 1. Get lengths of all files

In [ ]:
with open(ALL_SPLITS) as all_splits_file:
    all_splits = json.load(all_splits_file)

file_lengths = {}
for key in ['train', 'test']:
    split = all_splits[key]
    for split_n, filename in tqdm(split, total = len(split)):
        sub_path = os.path.join(split_n, filename)
        filepath = os.path.join(SPLITS_FOLDER, sub_path)
        metadata = parse_metadata(filepath)
        file_lengths[filename] = metadata.duration_s

with open(MANUAL_FILE_LIST, 'w') as output:
    json.dump(file_lengths, output)

## 2. Process Raven Selection

In [ ]:
# Determine whether or not a peak is within BOUNDARY_BUFFER of a file boundary
# if so, it will be excluded from global statistics regarding peak duration
def is_boundary_peak(row):
    file_end = row['file_length_sec']

    if row['dp_s_sec'] < BOUNDARY_BUFFER:
        return True

    if (file_end - row['dp_e_sec']) < BOUNDARY_BUFFER:
        return True

    return False

In [ ]:
raven_selection = pd.read_csv(RAVEN_SEL_PATH, sep = '\t')

df_peaks = raven_selection[raven_selection['View'] == 'Waveform 1']
df_peaks = df_peaks.drop(labels = ['Selection', 'View', 'Channel', 'Low Freq (Hz)', 'High Freq (Hz)'], axis = 1)

df_peaks['filename'] = df_peaks['Begin Path'].apply(lambda x : os.path.basename(x))
df_peaks['dp_s_sec'] = df_peaks['File Offset (s)'] 
df_peaks['dp_e_sec'] = (df_peaks['File Offset (s)'] + df_peaks['End Time (s)'] - df_peaks['Begin Time (s)']) 
df_peaks['dp_s'] = df_peaks['File Offset (s)'] * SAMPLE_RATE
df_peaks['dp_e'] = (df_peaks['File Offset (s)'] + df_peaks['End Time (s)'] - df_peaks['Begin Time (s)']) * SAMPLE_RATE

df_peaks['dp_s'] = df_peaks['dp_s'].astype(int)
df_peaks['dp_e'] = df_peaks['dp_e'].astype(int)

df_all_labels = df_peaks.drop(labels = ['Begin Time (s)', 'End Time (s)', 'Begin Path', 'File Offset (s)'], axis = 1)

with open(MANUAL_FILE_LIST) as lengths_file:
    file_lengths = json.load(lengths_file)

df_all_labels['duration_sec'] = df_all_labels['dp_e_sec'] - df_all_labels['dp_s_sec']
df_all_labels['file_length_sec'] = df_all_labels['filename'].apply(lambda x : file_lengths[x])
df_all_labels['is_boundary_peak'] = df_all_labels[['dp_s_sec', 'dp_e_sec', 'file_length_sec']].apply(is_boundary_peak, axis = 1)

## 3. Analysis

In [ ]:
print('Total number of events: ', len(df_all_labels))

df_valid_length_events = df_all_labels[~df_all_labels['is_boundary_peak']]
print('Total number of measurable events (not on file boundary): ', len(df_valid_length_events))
print('Minimum length event: ',min(df_valid_length_events['duration_sec']))
print('Maximum length event: ',max(df_valid_length_events['duration_sec']))

plt.style.use('ggplot')
plt.hist(df_valid_length_events['duration_sec'], bins = 50, color = 'cadetblue')
plt.show()